# NutriVision - Huan luyen 131 mon an (Colab T4)

Notebook nay tao model TFLite cho 131 lop, chay tren may (khong can mang khi
nhan dien):

- **101 mon Viet** tu `jeandeanny/vn-food-combined-2026` (40.432 anh, split da loc leak).
- **30 mon quoc te** tu `ethz/food101`.
- Danh gia tren **test set that**, bao cao top-1 / top-3 / per-class.
- Xuat `model.tflite` + `labels.txt` + `model_meta.json` de dua vao Android sau.

**Thoi gian du kien:** tai dataset 25-40 phut, train 2-3 gio voi GPU T4.

> Chay lan luot tung cell. Cell 7 xoa anh trung byte giua cac split - bat buoc
> chay truoc khi train, neu khong diem test se dep gia.
> Neu Colab ngat ket noi, dataset trong `/content/data` se mat va phai chay lai tu cell 6.


In [ ]:
# 1. Kiem tra GPU + cai thu vien phu tro
!nvidia-smi
# Colab da co san TensorFlow, khong cai lai de tranh lech phien ban CUDA
!pip install -q datasets pillow tqdm scikit-learn matplotlib

import tensorflow as tf, datasets, sklearn, PIL
print('tensorflow:', tf.__version__)
print('datasets  :', datasets.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPU       :', gpus if gpus else 'KHONG CO GPU - doi Runtime sang T4')

In [ ]:
# 2. Mount Google Drive - noi luu ket qua cuoi cung (cell 12)
from google.colab import drive
drive.mount('/content/drive')
OUT_DIR = '/content/drive/MyDrive/NutriVision_AI'
import os
os.makedirs(OUT_DIR, exist_ok=True)
print('Ket qua se luu tai:', OUT_DIR)

In [ ]:
%%writefile train_nutrivision.py
# 3. Script pipeline (khong sua truc tiep trong cell nay)
# %% [markdown]
# # NutriVision AI - Pipeline 131 lop (Colab T4)
#
# Pipeline nay thay the notebook cu:
# - Tai that dataset (khong phai `os.makedirs` roi train tren thu muc rong).
# - 101 mon Viet tu `jeandeanny/vn-food-combined-2026`.
# - 30 mon quoc te tu `ethz/food101`.
# - Gop 2 cap nhan trung, kiem tra du lop, chia train/val/test theo split goc.
# - Danh gia tren test set that, bao cao top-1 / top-3 / per-class.
# - Xuat `model.tflite` + `labels.txt` + `model_meta.json`.
#
# Chay lan luot:
# ```bash
# python train_nutrivision.py prepare          # tai + giai nen (~25-40 phut)
# python train_nutrivision.py dedupe           # xoa anh trung byte giua cac split
# python train_nutrivision.py train            # huan luyen (~2-3 gio T4)
# python train_nutrivision.py verify           # kiem tra TFLite
# ```
# Hoac `python train_nutrivision.py all` de chay ca 3 buoc.

# %%
"""NutriVision 131-class training pipeline (TensorFlow / TFLite)."""

import argparse
import hashlib
import json
import random
import sys
from pathlib import Path

HERE = Path(__file__).resolve().parent
VN_REPO = "jeandeanny/vn-food-combined-2026"
INTL_REPO = "ethz/food101"
IMG_SIZE = 224
SEED = 42
THRESHOLD = 0.70          # nguong tin cay de hien thi top-1, duoi nguong -> top-3
TOP_K = 3

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")


# ---------------------------------------------------------------- helpers
def log(msg):
    print(f"[NutriVision] {msg}", flush=True)


def load_class_map():
    path = HERE / "class_map.json"
    if not path.exists():
        raise SystemExit(
            "Thieu class_map.json. Chay: python build_classes.py"
        )
    with open(path, encoding="utf-8") as f:
        data = json.load(f)
    return data


def ensure_dir(path):
    Path(path).mkdir(parents=True, exist_ok=True)


def save_image(img, path, max_side=512):
    """Luu JPEG, giu ti le, gioi han canh dai."""
    from PIL import Image

    if not isinstance(img, Image.Image):
        img = Image.fromarray(img)
    img = img.convert("RGB")
    if max(img.size) > max_side:
        resample = getattr(Image, "Resampling", Image).LANCZOS
        img.thumbnail((max_side, max_side), resample)
    img.save(path, "JPEG", quality=90)


# ---------------------------------------------------------------- prepare
def prepare_vietnamese(data_dir, max_train, max_side, resume=True):
    """Tai 101 lop Viet tu Hugging Face va ghi ra data/{train,val,test}/<id>."""
    from datasets import load_dataset
    from tqdm.auto import tqdm

    class_map = load_class_map()
    hf_to_id = class_map["vietnamese"]

    split_pairs = [
        ("train", "train", max_train),
        ("validation", "val", None),
        ("test", "test", None),
    ]

    for hf_split, out_split, cap in split_pairs:
        out_root = Path(data_dir) / out_split
        ensure_dir(out_root)
        log(f"VN / {hf_split} -> {out_split}")
        ds = load_dataset(VN_REPO, split=hf_split)
        names = ds.features["label"].names

        counters = {}
        skipped = 0
        # chi doc cot label (nhanh), chi giai ma anh cua dong duoc giu lai
        label_col = ds["label"]
        for i, label_id in enumerate(
            tqdm(label_col, total=len(label_col), desc=f"vn-{out_split}", unit="img")
        ):
            hf_name = names[label_id]
            if hf_name not in hf_to_id:
                skipped += 1
                continue
            cid = hf_to_id[hf_name]
            n = counters.get(cid, 0)
            if cap and n >= cap:
                continue

            target_dir = out_root / cid
            ensure_dir(target_dir)
            target = target_dir / f"{i:06d}.jpg"
            if resume and target.exists():
                counters[cid] = n + 1
                continue

            save_image(ds[i]["image"], target, max_side)
            counters[cid] = n + 1

        log(f"  {sum(counters.values())} anh, {len(counters)} lop, bo qua {skipped}")


def prepare_international(data_dir, max_train, max_side, max_eval, resume=True):
    """Tai 30 lop quoc te tu Food-101. validation duoc tach doi val/test."""
    from datasets import load_dataset
    from tqdm.auto import tqdm

    class_map = load_class_map()
    intl_map = class_map["international"]          # app_id -> food101 class
    f101_to_id = {v: k for k, v in intl_map.items()}

    # ---- train
    out_root = Path(data_dir) / "train"
    ensure_dir(out_root)
    log("Food-101 / train -> train")
    ds = load_dataset(INTL_REPO, split="train")
    names = ds.features["label"].names
    counters = {}
    label_col = ds["label"]
    for i, label_id in enumerate(
        tqdm(label_col, total=len(label_col), desc="intl-train", unit="img")
    ):
        f101_name = names[label_id]
        if f101_name not in f101_to_id:
            continue
        cid = f101_to_id[f101_name]
        n = counters.get(cid, 0)
        if max_train and n >= max_train:
            continue
        target_dir = out_root / cid
        ensure_dir(target_dir)
        target = target_dir / f"f101_{i:06d}.jpg"
        if resume and target.exists():
            counters[cid] = n + 1
            continue
        save_image(ds[i]["image"], target, max_side)
        counters[cid] = n + 1
    log(f"  {sum(counters.values())} anh, {len(counters)} lop")

    # ---- validation cua Food-101: chan -> val, le -> test
    log("Food-101 / validation -> val + test (50/50)")
    ds = load_dataset(INTL_REPO, split="validation")
    names = ds.features["label"].names
    counters = {}
    label_col = ds["label"]
    for i, label_id in enumerate(
        tqdm(label_col, total=len(label_col), desc="intl-eval", unit="img")
    ):
        f101_name = names[label_id]
        if f101_name not in f101_to_id:
            continue
        cid = f101_to_id[f101_name]
        out_split = "val" if i % 2 == 0 else "test"
        n = counters.get((out_split, cid), 0)
        if max_eval and n >= max_eval:
            continue
        target_dir = Path(data_dir) / out_split / cid
        ensure_dir(target_dir)
        target = target_dir / f"f101_{i:06d}.jpg"
        if resume and target.exists():
            counters[(out_split, cid)] = n + 1
            continue
        save_image(ds[i]["image"], target, max_side)
        counters[(out_split, cid)] = n + 1

    for split in ("val", "test"):
        log(f"  {split}: {sum(v for (s, _), v in counters.items() if s == split)} anh")


def validate_tree(data_dir, min_per_class=1):
    """Kiem tra du lop, du split, khong trung file giua cac lop."""
    class_map = load_class_map()
    expected = {c["id"] for c in class_map["classes"]}
    problems = []

    for split in ("train", "val", "test"):
        root = Path(data_dir) / split
        if not root.exists():
            problems.append(f"Thieu thu muc {root}")
            continue
        actual = {p.name for p in root.iterdir() if p.is_dir()}
        missing = expected - actual
        extra = actual - expected
        if missing:
            problems.append(f"{split}: thieu {len(missing)} lop: {sorted(missing)[:8]}")
        if extra:
            problems.append(f"{split}: {len(extra)} lop la: {sorted(extra)[:8]}")
        for cid in sorted(expected & actual):
            n = len(list((root / cid).glob("*.jpg")))
            if n < min_per_class:
                problems.append(f"{split}/{cid}: chi {n} anh")

    if problems:
        for p in problems[:30]:
            log("LOI: " + p)
        raise SystemExit(f"Dataset chua hop le ({len(problems)} loi). Xem log phia tren.")

    log("Dataset hop le: du 131 lop o ca 3 split.")


SPLIT_PRIORITY = {"test": 0, "val": 1, "train": 2}


def find_duplicate_groups(data_dir, sample_limit=None):
    """Nhom cac file .jpg co noi dung byte giong het nhau.

    File trung byte giua hai split la ro ri du lieu: anh nam trong test ma
    cung nam trong train thi diem test khong con dang tin.
    """
    data_dir = Path(data_dir)
    files = sorted(data_dir.rglob("*.jpg"))
    if sample_limit:
        files = files[:sample_limit]
    seen = {}
    for p in files:
        h = hashlib.md5(p.read_bytes()).hexdigest()
        seen.setdefault(h, []).append(p)
    return [paths for paths in seen.values() if len(paths) > 1]


def check_duplicates(data_dir, sample_limit=None):
    """Bao cao file trung byte. Khong xoa gi."""
    data_dir = Path(data_dir)
    groups = find_duplicate_groups(data_dir, sample_limit)
    if groups:
        n_files = sum(len(g) - 1 for g in groups)
        log(f"CANH BAO: {len(groups)} nhom file trung byte ({n_files} file thua)")
        for g in groups[:10]:
            log("  " + " == ".join(str(p.relative_to(data_dir)) for p in g))
    else:
        log("Khong co file trung byte.")
    return groups


def dedupe_tree(data_dir, dry_run=False):
    """Xoa file trung byte, giu ban o split uu tien test > val > train.

    Uu tien giu test vi test la tap de do diem: giu nguyen test thi diem do
    khong bi lech, chi mat vai anh trong train. Khong bao gio xoa file cuoi
    cung cua mot lop (se lam lop do rong).
    """
    data_dir = Path(data_dir)
    groups = find_duplicate_groups(data_dir)
    if not groups:
        log("Khong co file trung byte.")
        return []

    counts = {}
    for split in ("train", "val", "test"):
        root = data_dir / split
        if not root.exists():
            continue
        for cid_dir in root.iterdir():
            if cid_dir.is_dir():
                counts[(split, cid_dir.name)] = len(list(cid_dir.glob("*.jpg")))

    def sort_key(p):
        rel = p.relative_to(data_dir)
        return (SPLIT_PRIORITY.get(rel.parts[0], 99), str(p))

    removed, kept_back = [], []
    for group in groups:
        ordered = sorted(group, key=sort_key)
        keeper = ordered[0]
        for p in ordered[1:]:
            rel = p.relative_to(data_dir)
            split, cid = rel.parts[0], rel.parts[1]
            if counts.get((split, cid), 0) <= 1:
                kept_back.append(p)
                log(f"  GIU LAI {rel} (xoa se lam rong lop {split}/{cid})")
                continue
            log(f"  {'se xoa' if dry_run else 'da xoa'} {rel} (giu {keeper.relative_to(data_dir)})")
            if not dry_run:
                p.unlink()
            counts[(split, cid)] -= 1
            removed.append(p)

    log(f"Trung byte: {len(groups)} nhom | {'se xoa' if dry_run else 'da xoa'} "
        f"{len(removed)} file | giu lai {len(kept_back)} file")
    if kept_back:
        log(f"CANH BAO: {len(kept_back)} file trung byte khong xoa duoc vi se lam rong lop.")
    return removed


def cmd_dedupe(args):
    dedupe_tree(args.data_dir, dry_run=args.dry_run)
    if args.dry_run:
        return
    validate_tree(args.data_dir, min_per_class=1)
    remaining = check_duplicates(args.data_dir)
    if remaining:
        raise SystemExit(
            "Van con file trung byte. Xem log, xu ly tay roi chay lai 'dedupe'."
        )


def find_cross_split_duplicates(data_dir):
    """Nhom file trung byte nam o >= 2 split khac nhau (ro ri du lieu)."""
    data_dir = Path(data_dir)
    leaks = []
    for group in find_duplicate_groups(data_dir):
        splits = {p.relative_to(data_dir).parts[0] for p in group}
        if len(splits) > 1:
            leaks.append(group)
    return leaks


def cmd_prepare(args):
    ensure_dir(args.data_dir)
    log(f"Data dir: {args.data_dir}")
    if not args.skip_vn:
        prepare_vietnamese(args.data_dir, args.max_train, args.max_side)
    if not args.skip_intl:
        prepare_international(args.data_dir, args.max_train, args.max_side, args.max_eval)
    validate_tree(args.data_dir, min_per_class=1)
    if args.check_duplicates:
        check_duplicates(args.data_dir)
    if getattr(args, "fix_duplicates", False):
        groups = find_duplicate_groups(args.data_dir)
        if groups:
            dedupe_tree(args.data_dir)
            validate_tree(args.data_dir, min_per_class=1)
            check_duplicates(args.data_dir)


# ---------------------------------------------------------------- train
def build_datasets(data_dir, batch_size, shuffle_seed):
    import tensorflow as tf

    train_dir = Path(data_dir) / "train"
    val_dir = Path(data_dir) / "val"
    test_dir = Path(data_dir) / "test"

    train_ds = tf.keras.utils.image_dataset_from_directory(
        train_dir, image_size=(IMG_SIZE, IMG_SIZE), batch_size=batch_size,
        label_mode="categorical", shuffle=True, seed=shuffle_seed,
    )
    val_ds = tf.keras.utils.image_dataset_from_directory(
        val_dir, image_size=(IMG_SIZE, IMG_SIZE), batch_size=batch_size,
        label_mode="categorical", shuffle=False,
    )
    test_ds = tf.keras.utils.image_dataset_from_directory(
        test_dir, image_size=(IMG_SIZE, IMG_SIZE), batch_size=batch_size,
        label_mode="categorical", shuffle=False,
    )

    class_names = train_ds.class_names
    assert val_ds.class_names == class_names, "Thu tu lop val khac train"
    assert test_ds.class_names == class_names, "Thu tu lop test khac train"
    return train_ds, val_ds, test_ds, class_names


def build_model(num_classes, arch):
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers

    if arch == "small":
        base = keras.applications.MobileNetV3Small(
            input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights="imagenet"
        )
        dropout = 0.2
    else:
        base = keras.applications.MobileNetV3Large(
            input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights="imagenet"
        )
        dropout = 0.3

    base.trainable = False

    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), dtype="float32")
    # dua preprocess vao trong model -> TFLite nhan thang float32 [0, 255]
    x = keras.applications.mobilenet_v3.preprocess_input(inputs)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(dropout)(x)
    outputs = layers.Dense(num_classes, activation="softmax", dtype="float32")(x)

    model = keras.Model(inputs, outputs, name=f"NutriVision_{arch}")
    return model, base


def compile_model(model, lr):
    from tensorflow import keras

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss="categorical_crossentropy",
        metrics=[
            "accuracy",
            keras.metrics.TopKCategoricalAccuracy(k=TOP_K, name="top3"),
        ],
    )


def cmd_train(args):
    import numpy as np
    import tensorflow as tf
    from tensorflow import keras

    tf.keras.utils.set_random_seed(SEED)
    random.seed(SEED)

    out_dir = Path(args.out_dir)
    ensure_dir(out_dir)

    # Chan train khi con anh trung byte giua cac split: diem test se dep gia.
    leaks = find_cross_split_duplicates(args.data_dir)
    if leaks:
        log(f"LOI: {len(leaks)} nhom anh trung byte giua cac split (ro ri du lieu):")
        for g in leaks[:10]:
            log("  " + " == ".join(str(p.relative_to(args.data_dir)) for p in g))
        raise SystemExit(
            "Chay truoc: python train_nutrivision.py dedupe --data-dir "
            f"{args.data_dir}"
        )
    log("Khong co anh trung byte giua cac split.")

    log("Nap dataset...")
    train_ds, val_ds, test_ds, class_names = build_datasets(
        args.data_dir, args.batch, SEED
    )
    log(f"{len(class_names)} lop | train {train_ds.cardinality().numpy()} batch | "
        f"val {val_ds.cardinality().numpy()} | test {test_ds.cardinality().numpy()}")

    expected = [line.strip() for line in (HERE / "labels_131.txt").read_text(encoding="utf-8").splitlines() if line.strip()]
    if sorted(expected) != sorted(class_names):
        missing = set(expected) - set(class_names)
        extra = set(class_names) - set(expected)
        raise SystemExit(f"Lech nhan! Thieu: {missing}. Thua: {extra}")

    # Luu thu tu lop ngay tu dau - day la thu tu model xuat ra
    (out_dir / "labels.txt").write_text("\n".join(class_names) + "\n", encoding="utf-8")

    AUTOTUNE = tf.data.AUTOTUNE

    augment = keras.Sequential(
        [
            keras.layers.RandomFlip("horizontal"),
            keras.layers.RandomRotation(0.08),
            keras.layers.RandomZoom(0.10),
        ],
        name="augment",
    )

    def to_float(images, labels):
        return tf.cast(images, tf.float32), labels

    def augment_batch(images, labels):
        return augment(images, training=True), labels

    train_ds = (
        train_ds
        .map(to_float, num_parallel_calls=AUTOTUNE)
        .map(augment_batch, num_parallel_calls=AUTOTUNE)
        .prefetch(AUTOTUNE)
    )
    val_ds = val_ds.map(to_float, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
    test_ds = test_ds.map(to_float, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)

    log(f"Xay model MobileNetV3-{args.arch} cho {len(class_names)} lop...")
    model, base = build_model(len(class_names), args.arch)
    model.summary()

    callbacks = [
        keras.callbacks.ModelCheckpoint(
            filepath=str(out_dir / "best.keras"),
            monitor="val_accuracy", save_best_only=True, verbose=1,
        ),
        keras.callbacks.EarlyStopping(
            monitor="val_accuracy", patience=3, restore_best_weights=True, verbose=1,
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6, verbose=1,
        ),
        keras.callbacks.CSVLogger(str(out_dir / "history.csv")),
    ]

    epochs1 = 1 if args.smoke else args.epochs1
    epochs2 = 0 if args.smoke else args.epochs2

    log(f"Giai doan 1: feature extraction ({epochs1} epoch)")
    compile_model(model, 1e-3)
    h1 = model.fit(
        train_ds, validation_data=val_ds, epochs=epochs1, callbacks=callbacks,
        steps_per_epoch=20 if args.smoke else None,
        validation_steps=10 if args.smoke else None,
    )

    if epochs2:
        log(f"Giai doan 2: fine-tuning ({epochs2} epoch)")
        base.trainable = True
        for layer in base.layers[:-40]:
            layer.trainable = False
        for layer in base.layers:
            if isinstance(layer, keras.layers.BatchNormalization):
                layer.trainable = False
        compile_model(model, 1e-4)
        h2 = model.fit(
            train_ds, validation_data=val_ds, epochs=epochs2, callbacks=callbacks,
        )
        history = {k: h1.history[k] + h2.history[k] for k in h1.history}
    else:
        history = h1.history

    best_path = out_dir / "best.keras"
    if best_path.exists():
        log("Nap lai trong so tot nhat tren validation...")
        model = keras.models.load_model(best_path)

    # ---- danh gia tren test set that
    log("Danh gia tren test set...")
    eval_ds = test_ds.take(10) if args.smoke else test_ds
    test_loss, test_acc, test_top3 = model.evaluate(eval_ds, verbose=0)
    log(f"  loss={test_loss:.4f} | acc={test_acc:.4f} | top3={test_top3:.4f}")

    log("Du doan toan bo test set de bao cao per-class...")
    y_prob = model.predict(eval_ds, verbose=1)
    y_true = np.concatenate([y.numpy() for _, y in eval_ds], axis=0)
    y_true_idx = y_true.argmax(axis=1)
    y_pred_idx = y_prob.argmax(axis=1)
    top3_idx = np.argsort(-y_prob, axis=1)[:, :TOP_K]
    top3_acc = float(np.mean([t in row for t, row in zip(y_true_idx, top3_idx)]))

    from sklearn.metrics import classification_report, confusion_matrix

    # labels=... : ep du 131 lop. Neu thieu (vd smoke chi cham 320 anh) thi
    # sklearn tu doan so lop tu y_true -> lech voi target_names.
    report = classification_report(
        y_true_idx, y_pred_idx, labels=list(range(len(class_names))),
        target_names=class_names, zero_division=0, digits=4
    )
    (out_dir / "classification_report.txt").write_text(report, encoding="utf-8")
    log(f"  top-1 = {float(np.mean(y_pred_idx == y_true_idx)):.4f} | top-3 = {top3_acc:.4f}")

    # ---- phan tich do tin cay: chon nguong cho Gemini fallback
    maxp = y_prob.max(axis=1)
    correct_mask = y_pred_idx == y_true_idx

    def quantiles(values):
        if len(values) == 0:
            return {}
        return {
            f"p{int(p * 100)}": float(np.quantile(values, p))
            for p in (0.10, 0.25, 0.50, 0.75, 0.90)
        }

    threshold_table = []
    for t in (0.40, 0.50, 0.60, 0.70, 0.80, 0.90):
        above = maxp >= t
        threshold_table.append({
            "threshold": t,
            "coverage": float(above.mean()),
            "accuracy_above": float(correct_mask[above].mean()) if above.any() else None,
            "accuracy_below": float(correct_mask[~above].mean()) if (~above).any() else None,
        })

    confidence = {
        "max_prob": {
            "all": quantiles(maxp),
            "correct": quantiles(maxp[correct_mask]),
            "incorrect": quantiles(maxp[~correct_mask]),
        },
        "threshold_table": threshold_table,
        "chosen_threshold": THRESHOLD,
        "note": (
            "coverage = ti le anh co max(p) >= nguong. "
            "accuracy_above = do dung trong nhom do (neu hien top-1 thi day la do chinh xac). "
            "Neu accuracy_above thap hoac coverage = 1.0, nguong nay khong co tac dung."
        ),
    }
    (out_dir / "confidence_report.json").write_text(
        json.dumps(confidence, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    for row in threshold_table:
        acc = row["accuracy_above"]
        acc_txt = "n/a" if acc is None else f"{acc:.2%}"
        log(f"  nguong {row['threshold']:.2f}: coverage={row['coverage']:.2%} "
            f"| dung khi tren nguong={acc_txt}")

    cm = confusion_matrix(y_true_idx, y_pred_idx, labels=list(range(len(class_names))))
    np.save(out_dir / "confusion_matrix.npy", cm)

    # ---- bieu do
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        axes[0].plot(history["accuracy"], label="Train", color="#10B981")
        axes[0].plot(history["val_accuracy"], label="Validation", color="#EF4444")
        axes[0].set_title("Accuracy")
        axes[0].set_xlabel("Epoch")
        axes[0].legend()
        axes[0].grid(alpha=0.3)

        axes[1].plot(history["loss"], label="Train", color="#10B981")
        axes[1].plot(history["val_loss"], label="Validation", color="#EF4444")
        axes[1].set_title("Loss")
        axes[1].set_xlabel("Epoch")
        axes[1].legend()
        axes[1].grid(alpha=0.3)

        fig.tight_layout()
        fig.savefig(out_dir / "training_curves.png", dpi=200)
        plt.close(fig)

        fig, ax = plt.subplots(figsize=(18, 18))
        im = ax.imshow(cm, interpolation="nearest", cmap="Blues")
        ax.set_title("Confusion matrix (test)")
        ax.set_xlabel("Predicted")
        ax.set_ylabel("True")
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        fig.tight_layout()
        fig.savefig(out_dir / "confusion_matrix.png", dpi=150)
        plt.close(fig)

        fig, ax = plt.subplots(figsize=(9, 5))
        bins = np.linspace(0, 1, 21)
        ax.hist(maxp[correct_mask], bins=bins, alpha=0.65, label="Correct", color="#10B981")
        ax.hist(maxp[~correct_mask], bins=bins, alpha=0.65, label="Wrong", color="#EF4444")
        ax.axvline(THRESHOLD, color="#111827", linestyle="--", label=f"Threshold {THRESHOLD}")
        ax.set_title("Confidence on test set (max softmax)")
        ax.set_xlabel("max(p)")
        ax.set_ylabel("Images")
        ax.legend()
        ax.grid(alpha=0.3)
        fig.tight_layout()
        fig.savefig(out_dir / "confidence_hist.png", dpi=160)
        plt.close(fig)
        log("  Da luu training_curves.png + confusion_matrix.png + confidence_hist.png")
    except Exception as exc:      # bieu do khong bat buoc
        log(f"  Bo qua bieu do: {exc}")

    # ---- xuat TFLite (dynamic-range int8, input/output float32)
    log("Xuat TFLite...")
    export_tflite(model, out_dir, class_names, int8_input=args.int8_input, rep_ds=val_ds)

    def count_images(split):
        return sum(1 for _ in (Path(args.data_dir) / split).rglob("*.jpg"))

    metrics = {
        "model": f"MobileNetV3-{args.arch}",
        "num_classes": len(class_names),
        "image_size": IMG_SIZE,
        "preprocessing": "mobilenet_v3.preprocess_input -> [-1, 1]",
        "top1": float(np.mean(y_pred_idx == y_true_idx)),
        "top3": top3_acc,
        "test_metrics": {
            "loss": float(test_loss),
            "accuracy": float(test_acc),
            "top3": float(test_top3),
        },
        "threshold": THRESHOLD,
        "top_k": TOP_K,
        "splits": {
            "train": count_images("train"),
            "val": count_images("val"),
            "test": count_images("test"),
        },
    }
    (out_dir / "metrics.json").write_text(
        json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    log(f"Xong. Ket qua trong {out_dir}")


def export_tflite(model, out_dir, class_names, int8_input=False, rep_ds=None):
    import numpy as np
    import tensorflow as tf

    saved_model_dir = Path(out_dir) / "saved_model"
    try:
        model.export(saved_model_dir)
        converter = tf.lite.TFLiteConverter.from_saved_model(str(saved_model_dir))
    except Exception as exc:
        log(f"  model.export loi ({exc}), dung from_keras_model")
        converter = tf.lite.TFLiteConverter.from_keras_model(model)

    converter.optimizations = [tf.lite.Optimize.DEFAULT]

    if int8_input:
        if rep_ds is None:
            raise SystemExit("int8-input can representative dataset (rep_ds).")

        def representative_data():
            for images, _ in rep_ds.take(20):
                for i in range(min(4, images.shape[0])):
                    # generator tra ve float32 goc [0,255]; converter tu luong tu hoa
                    yield [tf.cast(images[i:i + 1], tf.float32)]

        converter.representative_dataset = representative_data
        converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
        converter.inference_input_type = tf.uint8
        converter.inference_output_type = tf.uint8
        name = "model_int8.tflite"
    else:
        name = "model.tflite"

    tflite_model = converter.convert()
    path = Path(out_dir) / name
    path.write_bytes(tflite_model)
    log(f"  {name}: {len(tflite_model) / 1024 / 1024:.2f} MB")

    meta = {
        "file": name,
        "input_size": IMG_SIZE,
        "input_type": "uint8" if int8_input else "float32",
        "input_range": "0-255",
        "input_layout": "NHWC",
        "preprocessing": (
            "input uint8 0-255, model da quantize"
            if int8_input
            else "input float32 0-255, preprocess MobileNetV3 nam trong model"
        ),
        "output_type": "uint8" if int8_input else "float32",
        "num_classes": len(class_names),
        "threshold": THRESHOLD,
        "top_k": TOP_K,
    }
    (Path(out_dir) / "model_meta.json").write_text(
        json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8"
    )


# ---------------------------------------------------------------- verify
def cmd_verify(args):
    import numpy as np
    import tensorflow as tf

    out_dir = Path(args.out_dir)
    meta = json.loads((out_dir / "model_meta.json").read_text(encoding="utf-8"))
    class_names = (out_dir / "labels.txt").read_text(encoding="utf-8").splitlines()
    model_path = out_dir / meta["file"]

    interpreter = tf.lite.Interpreter(model_path=str(model_path))
    interpreter.allocate_tensors()
    inp = interpreter.get_input_details()[0]
    out = interpreter.get_output_details()[0]
    log(f"input  {inp['shape']} {inp['dtype']}")
    log(f"output {out['shape']} {out['dtype']}")

    assert inp["shape"][1] == meta["input_size"], "Sai kich thuoc input"
    assert out["shape"][-1] == len(class_names), "So lop output khac labels.txt"

    # chay thu 10 anh test, moi lop 1 anh de de soi hon
    test_dir = Path(args.data_dir) / "test"
    files = []
    for class_dir in sorted(p for p in test_dir.iterdir() if p.is_dir()):
        first = sorted(class_dir.glob("*.jpg"))[:1]
        files.extend(first)
        if len(files) >= 10:
            break
    if not files:
        log("Khong co anh test de chay thu.")
        return

    correct = 0
    correct_top3 = 0
    for p in files:
        img = tf.keras.utils.load_img(p, target_size=(meta["input_size"], meta["input_size"]))
        arr = tf.keras.utils.img_to_array(img)          # float32, 0-255
        if inp["dtype"] == np.uint8:
            arr = arr.astype(np.uint8)
        else:
            arr = arr.astype(np.float32)
        arr = np.expand_dims(arr, 0)

        interpreter.set_tensor(inp["index"], arr)
        interpreter.invoke()
        pred = interpreter.get_tensor(out["index"])[0]

        if out["dtype"] in (np.uint8, np.int8):
            scale, zero = out["quantization"]
            pred = (pred.astype(np.float32) - zero) * scale

        top = np.argsort(-pred)[:meta["top_k"]]
        true_id = p.parent.name
        if class_names[top[0]] == true_id:
            correct += 1
        if true_id in [class_names[i] for i in top]:
            correct_top3 += 1
        log(f"  {true_id}: " + ", ".join(f"{class_names[i]} {pred[i]*100:.1f}%" for i in top))

    log(f"Top-1 tren {len(files)} anh: {correct}/{len(files)} | top-3: {correct_top3}/{len(files)}")
    if correct < len(files) // 2:
        log("CANH BAO: ket qua thap - kiem tra lai thu tu labels va preprocessing.")


# ---------------------------------------------------------------- cli
def build_parser():
    p = argparse.ArgumentParser(description="NutriVision 131-class pipeline")
    sub = p.add_subparsers(dest="cmd", required=True)

    def add_common(sp):
        sp.add_argument("--data-dir", default="/content/data")
        sp.add_argument("--out-dir", default="/content/out")

    prep = sub.add_parser("prepare", help="tai + chuan hoa dataset")
    add_common(prep)
    prep.add_argument("--max-train", type=int, default=300, help="toi da anh/lop cho train")
    prep.add_argument("--max-eval", type=int, default=None, help="toi da anh/lop cho val/test")
    prep.add_argument("--max-side", type=int, default=512)
    prep.add_argument("--skip-vn", action="store_true")
    prep.add_argument("--skip-intl", action="store_true")
    prep.add_argument("--check-duplicates", action="store_true")
    prep.add_argument("--fix-duplicates", action="store_true",
                      help="xoa file trung byte ngay sau khi prepare (giu test > val > train)")
    prep.set_defaults(func=cmd_prepare)

    ded = sub.add_parser("dedupe", help="xoa file trung byte giua cac split")
    add_common(ded)
    ded.add_argument("--dry-run", action="store_true", help="chi in ra, khong xoa file")
    ded.set_defaults(func=cmd_dedupe)

    tr = sub.add_parser("train", help="huan luyen + xuat TFLite")
    add_common(tr)
    tr.add_argument("--arch", choices=["small", "large"], default="large")
    tr.add_argument("--batch", type=int, default=32)
    tr.add_argument("--epochs1", type=int, default=8)
    tr.add_argument("--epochs2", type=int, default=6)
    tr.add_argument("--int8-input", action="store_true",
                    help="full int8 (uint8 input) - nho hon nhung kho dung hon")
    tr.add_argument("--smoke", action="store_true", help="chay 1 epoch ngan de kiem tra")
    tr.set_defaults(func=cmd_train)

    ve = sub.add_parser("verify", help="kiem tra model.tflite")
    add_common(ve)
    ve.set_defaults(func=cmd_verify)

    allp = sub.add_parser("all", help="prepare + train + verify")
    add_common(allp)
    allp.add_argument("--max-train", type=int, default=300)
    allp.add_argument("--max-side", type=int, default=512)
    allp.add_argument("--arch", choices=["small", "large"], default="large")
    allp.add_argument("--batch", type=int, default=32)
    allp.add_argument("--epochs1", type=int, default=8)
    allp.add_argument("--epochs2", type=int, default=6)
    allp.add_argument("--int8-input", action="store_true")
    allp.add_argument("--smoke", action="store_true")
    allp.set_defaults(func=None)

    return p


def main():
    parser = build_parser()
    args = parser.parse_args()

    if args.cmd == "all":
        cmd_prepare(argparse.Namespace(
            data_dir=args.data_dir, out_dir=args.out_dir, max_train=args.max_train,
            max_eval=None, max_side=args.max_side, skip_vn=False, skip_intl=False,
            check_duplicates=False,
        ))
        cmd_train(args)
        cmd_verify(args)
    else:
        args.func(args)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile class_map.json
{
  "vietnamese": {
    "baba-nau-chuoi-dau": "baba_nau_chuoi_dau",
    "banh-bao": "banh_bao",
    "banh-beo": "banh_beo",
    "banh-bo": "banh_bo",
    "banh-bot-loc": "banh_bot_loc",
    "banh-can": "banh_can",
    "banh-canh": "banh_canh",
    "banh-chung": "banh_chung",
    "banh-cong": "banh_cong",
    "banh-cuon": "banh_cuon",
    "banh-da-cua": "banh_da_cua",
    "banh-da-lon": "banh_da_lon",
    "banh-duc": "banh_duc",
    "banh-gai": "banh_gai",
    "banh-giay": "banh_giay",
    "banh-gio": "banh_gio",
    "banh-hoi": "banh_hoi",
    "banh-khot": "banh_khot",
    "banh-la": "banh_la",
    "banh-mi": "banh_mi",
    "banh-mi-chao": "banh_mi_chao",
    "banh-pia": "banh_pia",
    "banh-tai-heo": "banh_tai_heo",
    "banh-tet": "banh_tet",
    "banh-tieu": "banh_tieu",
    "banh-tom-ho-tay": "banh_tom_ho_tay",
    "banh-trang-nuong": "banh_trang_nuong",
    "banh-troi-nuoc": "banh_troi_nuoc",
    "banh-trung-thu": "banh_trung_thu",
    "banh-u": "banh_u",
    "banh-xeo": "banh_xeo",
    "bo-kho": "bo_kho",
    "bo-la-lot": "bo_la_lot",
    "bo-luc-lac": "bo_luc_lac",
    "bo-ne": "bo_ne",
    "bo-nuong-la-lot": "bo_la_lot",
    "bun-bo-hue": "bun_bo_hue",
    "bun-cha": "bun_cha",
    "bun-cha-ca": "bun_cha_ca",
    "bun-dau-mam-tom": "bun_dau_mam_tom",
    "bun-mam": "bun_mam",
    "bun-rieu": "bun_rieu",
    "bun-thit-nuong": "bun_thit_nuong",
    "ca-kho-to": "ca_kho_to",
    "ca-loc-nuong": "ca_loc_nuong",
    "ca-muoi-xoi": "ca_muoi_xoi",
    "ca-ri-ga": "ca_ri_ga",
    "ca-sot-ca-chua": "ca_sot_ca_chua",
    "canh-bi-do": "canh_bi_do",
    "canh-chua": "canh_chua",
    "canh-cua": "canh_cua",
    "canh-kho-hoa": "canh_kho_hoa",
    "canh-khoai-tim": "canh_khoai_tim",
    "cao-lau": "cao_lau",
    "cha-ca-la-vong": "cha_ca_la_vong",
    "cha-com": "cha_com",
    "cha-lui": "cha_lui",
    "chao-long": "chao_long",
    "chao-vit": "chao_vit",
    "com-chay-cha-bong": "com_chay_cha_bong",
    "com-chien": "com_chien",
    "com-ga-xoi-mo": "com_ga_xoi_mo",
    "com-lam": "com_lam",
    "com-rang-dua-bo": "com_rang_dua_bo",
    "com-tam": "com_tam",
    "cua-hap-bia": "cua_hap_bia",
    "cut-lon-xao-me": "cut_lon_xao_me",
    "ga-chien-nuoc-mam": "ga_chien_nuoc_mam",
    "ga-hap-la-chanh": "ga_hap_la_chanh",
    "goi-ca-chich": "goi_ca_chich",
    "goi-cuon": "goi_cuon",
    "hu-tieu": "hu_tieu",
    "khau-nhuc": "khau_nhuc",
    "kho-muc-nuong": "kho_muc_nuong",
    "kho-quet": "kho_quet",
    "lap-xuong": "lap_xuong",
    "luon-om-chuoi-dau": "luon_om_chuoi_dau",
    "luon-xao-xa-ot": "luon_xao_xa_ot",
    "mam-chung": "mam_chung",
    "mam-tep-chung-thit": "mam_tep_chung_thit",
    "mi-quang": "mi_quang",
    "mi-xao-gion": "mi_xao_gion",
    "muc-nhoi-thit": "muc_nhoi_thit",
    "nam-pia": "banh_pia",
    "nem-chua": "nem_chua",
    "nem-nuong-nha-trang": "nem_nuong_nha_trang",
    "nui-xao": "nui_xao",
    "oc-buou-hap": "oc_buou_hap",
    "oc-huong-xao": "oc_huong_xao",
    "oc-len-xao-dua": "oc_len_xao_dua",
    "pho": "pho",
    "rau-muong-xao": "rau_muong_xao",
    "sup-cua": "sup_cua",
    "tau-hu-nhoi-thit": "tau_hu_nhoi_thit",
    "tau-hu-non": "tau_hu_non",
    "thit-dong": "thit_dong",
    "thit-kho-tau": "thit_kho_tau",
    "thit-trau-gac-bep": "thit_trau_gac_bep",
    "tiet-canh": "tiet_canh",
    "trung-vit-lon": "trung_vit_lon",
    "xoi-gac": "xoi_gac",
    "xoi-nep-than": "xoi_nep_than",
    "xoi-xeo": "xoi_xeo"
  },
  "international": {
    "pizza": "pizza",
    "hamburger": "hamburger",
    "hot_dog": "hot_dog",
    "french_fries": "french_fries",
    "club_sandwich": "club_sandwich",
    "spaghetti": "spaghetti_bolognese",
    "lasagna": "lasagna",
    "steak": "steak",
    "tacos": "tacos",
    "caesar_salad": "caesar_salad",
    "greek_salad": "greek_salad",
    "sushi": "sushi",
    "sashimi": "sashimi",
    "ramen": "ramen",
    "bibimbap": "bibimbap",
    "pad_thai": "pad_thai",
    "peking_duck": "peking_duck",
    "donut": "donuts",
    "waffles": "waffles",
    "pancakes": "pancakes",
    "chicken_wings": "chicken_wings",
    "fish_and_chips": "fish_and_chips",
    "nachos": "nachos",
    "onion_rings": "onion_rings",
    "macaroni_and_cheese": "macaroni_and_cheese",
    "grilled_salmon": "grilled_salmon",
    "dumplings": "dumplings",
    "takoyaki": "takoyaki",
    "tiramisu": "tiramisu",
    "cheesecake": "cheesecake"
  },
  "classes": [
    {
      "id": "baba_nau_chuoi_dau",
      "name_vi": "Ba ba nấu chuối đậu",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "banh_bao",
      "name_vi": "Bánh bao xá xíu trứng cút",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "banh_beo",
      "name_vi": "Bánh bèo chén tôm cháy",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "banh_bo",
      "name_vi": "Bánh bò",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "banh_bot_loc",
      "name_vi": "Bánh bột lọc tôm thịt",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "banh_can",
      "name_vi": "Bánh căn Phan Thiết",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "banh_canh",
      "name_vi": "Bánh canh cua ghẹ",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "banh_chung",
      "name_vi": "Bánh chưng Tết",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "banh_cong",
      "name_vi": "Bánh cống Sóc Trăng",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "banh_cuon",
      "name_vi": "Bánh cuốn nóng chả lụa",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "banh_da_cua",
      "name_vi": "Bánh đa cua Hải Phòng",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "banh_da_lon",
      "name_vi": "Bánh da lợn",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "banh_duc",
      "name_vi": "Bánh đúc",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "banh_gai",
      "name_vi": "Bánh gai",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "banh_giay",
      "name_vi": "Bánh giầy",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "banh_gio",
      "name_vi": "Bánh giò nóng nhân thịt nấm",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "banh_hoi",
      "name_vi": "Bánh hỏi",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "banh_khot",
      "name_vi": "Bánh khọt tôm Vũng Tàu",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "banh_la",
      "name_vi": "Bánh lá",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "banh_mi",
      "name_vi": "Bánh mì thịt pate Việt Nam",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "banh_mi_chao",
      "name_vi": "Bánh mì chảo thập cẩm",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "banh_pia",
      "name_vi": "Bánh pía sầu riêng trứng muối",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "banh_tai_heo",
      "name_vi": "Bánh tai heo",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "banh_tet",
      "name_vi": "Bánh tét miền Nam",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "banh_tieu",
      "name_vi": "Bánh tiêu mè",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "banh_tom_ho_tay",
      "name_vi": "Bánh tôm Hồ Tây",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "banh_trang_nuong",
      "name_vi": "Bánh tráng nướng Đà Lạt",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "banh_troi_nuoc",
      "name_vi": "Bánh trôi nước gừng đường thốt nốt",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "banh_trung_thu",
      "name_vi": "Bánh trung thu",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "banh_u",
      "name_vi": "Bánh ú",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "banh_xeo",
      "name_vi": "Bánh xèo miền Tây",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "bo_kho",
      "name_vi": "Bò kho bánh mì",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "bo_la_lot",
      "name_vi": "Bò nướng lá lốt",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "bo_luc_lac",
      "name_vi": "Bò lúc lắc khoai tây chiên",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "bo_ne",
      "name_vi": "Bò né trứng ốp la",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "bun_bo_hue",
      "name_vi": "Bún bò Huế",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "bun_cha",
      "name_vi": "Bún chả Hà Nội",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "bun_cha_ca",
      "name_vi": "Bún chả cá Đà Nẵng",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "bun_dau_mam_tom",
      "name_vi": "Bún đậu mắm tôm",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "bun_mam",
      "name_vi": "Bún mắm miền Tây",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "bun_rieu",
      "name_vi": "Bún riêu cua đồng",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "bun_thit_nuong",
      "name_vi": "Bún thịt nướng",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "ca_kho_to",
      "name_vi": "Cá kho tộ miền Tây",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "ca_loc_nuong",
      "name_vi": "Cá lóc nướng trui",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "ca_muoi_xoi",
      "name_vi": "Cá muối xổi",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "ca_ri_ga",
      "name_vi": "Cà ri gà",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "ca_sot_ca_chua",
      "name_vi": "Cá sốt cà chua",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "canh_bi_do",
      "name_vi": "Canh bí đỏ",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "canh_chua",
      "name_vi": "Canh chua cá lóc miền Tây",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "canh_cua",
      "name_vi": "Canh cua đồng rau đay mồng tơi",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "canh_kho_hoa",
      "name_vi": "Canh khổ qua",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "canh_khoai_tim",
      "name_vi": "Canh khoai tím",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "cao_lau",
      "name_vi": "Cao lầu Hội An",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "cha_ca_la_vong",
      "name_vi": "Chả cá Lã Vọng",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "cha_com",
      "name_vi": "Chả cốm",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "cha_lui",
      "name_vi": "Chả lụi",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "chao_long",
      "name_vi": "Cháo lòng",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "chao_vit",
      "name_vi": "Cháo vịt",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "com_chay_cha_bong",
      "name_vi": "Cơm cháy chà bông mỡ hành",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "com_chien",
      "name_vi": "Cơm chiên thập cẩm",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "com_ga_xoi_mo",
      "name_vi": "Cơm gà xối mỡ",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "com_lam",
      "name_vi": "Cơm lam nướng ống tre",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "com_rang_dua_bo",
      "name_vi": "Cơm rang dưa bò",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "com_tam",
      "name_vi": "Cơm tấm sườn bì chả",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "cua_hap_bia",
      "name_vi": "Cua biển hấp bia sả",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "cut_lon_xao_me",
      "name_vi": "Cút lộn xào me",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "ga_chien_nuoc_mam",
      "name_vi": "Gà chiên nước mắm",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "ga_hap_la_chanh",
      "name_vi": "Gà ta luộc lá chanh",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "goi_ca_chich",
      "name_vi": "Gỏi cá chích",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "goi_cuon",
      "name_vi": "Gỏi cuốn tôm thịt",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "hu_tieu",
      "name_vi": "Hủ tiếu Nam Vang",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "khau_nhuc",
      "name_vi": "Khâu nhục",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "kho_muc_nuong",
      "name_vi": "Khô mực nướng",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "kho_quet",
      "name_vi": "Kho quẹt chấm rau luộc",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "lap_xuong",
      "name_vi": "Lạp xưởng nướng than hoa",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "luon_om_chuoi_dau",
      "name_vi": "Lươn om chuối đậu",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "luon_xao_xa_ot",
      "name_vi": "Lươn xào sả ớt",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "mam_chung",
      "name_vi": "Mắm chưng",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "mam_tep_chung_thit",
      "name_vi": "Mắm tép chưng thịt",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "mi_quang",
      "name_vi": "Mì Quảng tôm thịt",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "mi_xao_gion",
      "name_vi": "Mì xào giòn",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "muc_nhoi_thit",
      "name_vi": "Mực nhồi thịt sốt cà",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "nem_chua",
      "name_vi": "Nem chua Thanh Hóa",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "nem_nuong_nha_trang",
      "name_vi": "Nem nướng Nha Trang",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "nui_xao",
      "name_vi": "Nui xào",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "oc_buou_hap",
      "name_vi": "Ốc bươu nhồi thịt hấp sả",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "oc_huong_xao",
      "name_vi": "Ốc hương xào bơ tỏi",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "oc_len_xao_dua",
      "name_vi": "Ốc len xào dừa",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "pho",
      "name_vi": "Phở",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "rau_muong_xao",
      "name_vi": "Rau muống xào tỏi",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "sup_cua",
      "name_vi": "Súp cua óc heo",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "tau_hu_nhoi_thit",
      "name_vi": "Tàu hũ nhồi thịt",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "tau_hu_non",
      "name_vi": "Tàu hũ non",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "thit_dong",
      "name_vi": "Thịt đông chân giò",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "thit_kho_tau",
      "name_vi": "Thịt kho tàu trứng cút",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "thit_trau_gac_bep",
      "name_vi": "Thịt trâu gác bếp Tây Bắc",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "tiet_canh",
      "name_vi": "Tiết canh",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "trung_vit_lon",
      "name_vi": "Trứng vịt lộn",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "xoi_gac",
      "name_vi": "Xôi gấc hạt sen",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "xoi_nep_than",
      "name_vi": "Xôi nếp than sữa dừa",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "xoi_xeo",
      "name_vi": "Xôi xéo mỡ hành đậu xanh",
      "group": "vietnamese",
      "source": "vn-food-combined-2026"
    },
    {
      "id": "bibimbap",
      "name_vi": "Cơm trộn thố đá Hàn Quốc (Bibimbap)",
      "group": "international",
      "source": "food-101"
    },
    {
      "id": "caesar_salad",
      "name_vi": "Salad Caesar sốt kem phô mai",
      "group": "international",
      "source": "food-101"
    },
    {
      "id": "cheesecake",
      "name_vi": "Bánh phô mai Cheesecake",
      "group": "international",
      "source": "food-101"
    },
    {
      "id": "chicken_wings",
      "name_vi": "Cánh gà chiên",
      "group": "international",
      "source": "food-101"
    },
    {
      "id": "club_sandwich",
      "name_vi": "Sandwich tam giác thịt nguội phô mai",
      "group": "international",
      "source": "food-101"
    },
    {
      "id": "donut",
      "name_vi": "Bánh vòng socola đường (Donut)",
      "group": "international",
      "source": "food-101"
    },
    {
      "id": "dumplings",
      "name_vi": "Sủi cảo hấp",
      "group": "international",
      "source": "food-101"
    },
    {
      "id": "fish_and_chips",
      "name_vi": "Cá và khoai tây chiên",
      "group": "international",
      "source": "food-101"
    },
    {
      "id": "french_fries",
      "name_vi": "Khoai tây chiên giòn",
      "group": "international",
      "source": "food-101"
    },
    {
      "id": "greek_salad",
      "name_vi": "Salad Hy Lạp phô mai Feta",
      "group": "international",
      "source": "food-101"
    },
    {
      "id": "grilled_salmon",
      "name_vi": "Cá hồi nướng",
      "group": "international",
      "source": "food-101"
    },
    {
      "id": "hamburger",
      "name_vi": "Burger bò phô mai (Cheeseburger)",
      "group": "international",
      "source": "food-101"
    },
    {
      "id": "hot_dog",
      "name_vi": "Bánh mì kẹp xúc xích Mỹ (Hot dog)",
      "group": "international",
      "source": "food-101"
    },
    {
      "id": "lasagna",
      "name_vi": "Mì Ý đút lò phô mai (Lasagna)",
      "group": "international",
      "source": "food-101"
    },
    {
      "id": "macaroni_and_cheese",
      "name_vi": "Nui phô mai",
      "group": "international",
      "source": "food-101"
    },
    {
      "id": "nachos",
      "name_vi": "Nachos phô mai",
      "group": "international",
      "source": "food-101"
    },
    {
      "id": "onion_rings",
      "name_vi": "Hành tây chiên vòng",
      "group": "international",
      "source": "food-101"
    },
    {
      "id": "pad_thai",
      "name_vi": "Phở xào chua ngọt Thái Lan (Pad Thai)",
      "group": "international",
      "source": "food-101"
    },
    {
      "id": "pancakes",
      "name_vi": "Bánh kếp nướng mật ong bơ (Pancakes)",
      "group": "international",
      "source": "food-101"
    },
    {
      "id": "peking_duck",
      "name_vi": "Vịt quay Bắc Kinh cuốn bánh tráng",
      "group": "international",
      "source": "food-101"
    },
    {
      "id": "pizza",
      "name_vi": "Pizza phô mai xúc xích Pepperoni",
      "group": "international",
      "source": "food-101"
    },
    {
      "id": "ramen",
      "name_vi": "Mì Ramen Nhật Bản thịt xá xíu Chashu",
      "group": "international",
      "source": "food-101"
    },
    {
      "id": "sashimi",
      "name_vi": "Sashimi cá hồi tươi Na Uy",
      "group": "international",
      "source": "food-101"
    },
    {
      "id": "spaghetti",
      "name_vi": "Mì Ý sốt bò băm (Spaghetti Bolognese)",
      "group": "international",
      "source": "food-101"
    },
    {
      "id": "steak",
      "name_vi": "Bít tết thăn bò sốt tiêu đen",
      "group": "international",
      "source": "food-101"
    },
    {
      "id": "sushi",
      "name_vi": "Sushi cơm cuộn cá hồi trứng cá (Nigiri/Maki)",
      "group": "international",
      "source": "food-101"
    },
    {
      "id": "tacos",
      "name_vi": "Bánh Tacos Mexico thịt bò giòn",
      "group": "international",
      "source": "food-101"
    },
    {
      "id": "takoyaki",
      "name_vi": "Bánh bạch tuộc Takoyaki",
      "group": "international",
      "source": "food-101"
    },
    {
      "id": "tiramisu",
      "name_vi": "Bánh Tiramisu",
      "group": "international",
      "source": "food-101"
    },
    {
      "id": "waffles",
      "name_vi": "Bánh quế nướng mật ong bơ (Waffles)",
      "group": "international",
      "source": "food-101"
    }
  ]
}

In [ ]:
%%writefile labels_131.txt
baba_nau_chuoi_dau
banh_bao
banh_beo
banh_bo
banh_bot_loc
banh_can
banh_canh
banh_chung
banh_cong
banh_cuon
banh_da_cua
banh_da_lon
banh_duc
banh_gai
banh_giay
banh_gio
banh_hoi
banh_khot
banh_la
banh_mi
banh_mi_chao
banh_pia
banh_tai_heo
banh_tet
banh_tieu
banh_tom_ho_tay
banh_trang_nuong
banh_troi_nuoc
banh_trung_thu
banh_u
banh_xeo
bibimbap
bo_kho
bo_la_lot
bo_luc_lac
bo_ne
bun_bo_hue
bun_cha
bun_cha_ca
bun_dau_mam_tom
bun_mam
bun_rieu
bun_thit_nuong
ca_kho_to
ca_loc_nuong
ca_muoi_xoi
ca_ri_ga
ca_sot_ca_chua
caesar_salad
canh_bi_do
canh_chua
canh_cua
canh_kho_hoa
canh_khoai_tim
cao_lau
cha_ca_la_vong
cha_com
cha_lui
chao_long
chao_vit
cheesecake
chicken_wings
club_sandwich
com_chay_cha_bong
com_chien
com_ga_xoi_mo
com_lam
com_rang_dua_bo
com_tam
cua_hap_bia
cut_lon_xao_me
donut
dumplings
fish_and_chips
french_fries
ga_chien_nuoc_mam
ga_hap_la_chanh
goi_ca_chich
goi_cuon
greek_salad
grilled_salmon
hamburger
hot_dog
hu_tieu
khau_nhuc
kho_muc_nuong
kho_quet
lap_xuong
lasagna
luon_om_chuoi_dau
luon_xao_xa_ot
macaroni_and_cheese
mam_chung
mam_tep_chung_thit
mi_quang
mi_xao_gion
muc_nhoi_thit
nachos
nem_chua
nem_nuong_nha_trang
nui_xao
oc_buou_hap
oc_huong_xao
oc_len_xao_dua
onion_rings
pad_thai
pancakes
peking_duck
pho
pizza
ramen
rau_muong_xao
sashimi
spaghetti
steak
sup_cua
sushi
tacos
takoyaki
tau_hu_nhoi_thit
tau_hu_non
thit_dong
thit_kho_tau
thit_trau_gac_bep
tiet_canh
tiramisu
trung_vit_lon
waffles
xoi_gac
xoi_nep_than
xoi_xeo


In [ ]:
# 6. Tai + chuan hoa dataset (khoang 25-40 phut, ~14 GB dia)
# --max-train 250 giup can bang 131 lop va giam thoi gian train.
# Cuoi buoc se bao cao file trung byte (neu co -> chay cell 7).
!python train_nutrivision.py prepare --data-dir /content/data --max-train 250 --check-duplicates

In [ ]:
# 7. Xoa file trung byte (bat buoc - tranh ro ri du lieu giua train/val/test)
# Giu ban o split uu tien test > val > train. Chay lai nhieu lan cung duoc.
!python train_nutrivision.py dedupe --data-dir /content/data

In [ ]:
# 8. SMOKE TEST - chay 1 epoch ngan de chac pipeline khong loi (2-3 phut)
# Bat buoc chay truoc khi train that.
!python train_nutrivision.py train --data-dir /content/data --out-dir /content/out_smoke --arch small --smoke

In [ ]:
# 9. Train that + xuat TFLite (2-3 gio T4)
# Ghi ra /content/out (dia cua Colab) cho nhanh; cuoi buoc se copy sang Drive.
# Nhanh hon: --arch small --epochs1 6 --epochs2 4
!python train_nutrivision.py train --data-dir /content/data --out-dir /content/out --arch large --epochs1 6 --epochs2 4

In [ ]:
# 10. Kiem tra model.tflite bang 10 anh test
!python train_nutrivision.py verify --data-dir /content/data --out-dir /content/out

In [ ]:
# 11. Xem ket qua + bang nguong tin cay (de chon nguong goi Gemini)
import json, os
OUT = '/content/out'
print(open(os.path.join(OUT, 'classification_report.txt'), encoding='utf-8').read()[:3000])
m = json.load(open(os.path.join(OUT, 'metrics.json'), encoding='utf-8'))
print('TOP-1:', m['top1'], '| TOP-3:', m['top3'])
c = json.load(open(os.path.join(OUT, 'confidence_report.json'), encoding='utf-8'))
for row in c['threshold_table']:
    acc = row['accuracy_above']
    print(f"nguong {row['threshold']:.2f} | coverage {row['coverage']:.1%} | dung khi tren nguong {acc if acc is None else format(acc, '.1%')}")

In [ ]:
# 12. Copy ket qua sang Google Drive + dong goi tai ve
import os, shutil
os.makedirs(OUT_DIR, exist_ok=True)
FILES = ['model.tflite', 'labels.txt', 'model_meta.json', 'metrics.json',
         'confidence_report.json', 'classification_report.txt',
         'training_curves.png', 'confusion_matrix.png', 'confidence_hist.png']
for name in FILES:
    src = os.path.join('/content/out', name)
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(OUT_DIR, name))
        print('saved  ', name)
    else:
        print('MISSING', name)

!cd /content/out && zip -r /content/nutrivision_model.zip model.tflite labels.txt model_meta.json metrics.json confidence_report.json training_curves.png confusion_matrix.png confidence_hist.png classification_report.txt
from google.colab import files
files.download('/content/nutrivision_model.zip')
print('Da luu tren Drive:', OUT_DIR)